# 01 Descriptive analysis: baseline patient profile (PB2)
Group 5 PROHI: Risk Factor Prediction of Chronic Kidney Disease

User: Clinical Data Analyst.

Question: what does the typical CKD patient look like compared with a non-CKD patient in this hospital?



In [ ]:
GITHUB_USER = "mhoyor"

%cd /content
!rm -rf ckd-risk-app
!git clone https://github.com/{GITHUB_USER}/ckd-risk-app.git
%cd /content/ckd-risk-app

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from data_prep import load_data, _bin_key
from viz import apply_style, grouped_bar, grouped_barh

apply_style()
os.makedirs("figures", exist_ok=True)

d = load_data() #load the data
raw = d["raw"].copy()
raw["group"] = raw["class"].map({"ckd": "CKD", "notckd": "Non-CKD"})

# Readable names for the variables
LAB_VARS = {
    "age": "Age (years)",
    "bp": "Diastolic blood pressure flag (0/1, as coded)",
    "bp limit": "Blood pressure category (0/1/2, as coded)",
    "bgr": "Random blood glucose (mg/dL)",
    "bu": "Blood urea (mg/dL)",
    "sc": "Serum creatinine (mg/dL)",
    "hemo": "Haemoglobin (g/dL)",
    "sg": "Urine specific gravity",
    "al": "Urine albumin (dipstick grade)",
}
CONDITIONS = {
    "htn": "Hypertension",
    "dm": "Diabetes",
    "cad": "Coronary artery disease",
    "ane": "Anaemia",
    "pe": "Pedal oedema",
    "appet": "Poor appetite",
}

# Tidy the albumin labels: "< 0" means grade 0, "1 - 1" means grade 1, and so on
AL_LABELS = {"< 0": "0", "1 - 1": "1", "2 - 2": "2", "3 - 3": "3", "≥ 4": "4+"}
raw["al"] = raw["al"].map(AL_LABELS)

def ordered_levels(col):
    if col == "al":
        return ["0", "1", "2", "3", "4+"]
    if col in ("bp", "bp limit"):
        return sorted(raw[col].unique())
    return sorted(raw[col].unique(), key=_bin_key)

## Step 1: Group sizes


In [ ]:
sizes = raw["group"].value_counts() #group the patients by diagnosis
pd.DataFrame({"Patients": sizes, "%": (100 * sizes / sizes.sum()).round(1)})

## Step 2: Count and percentage tables per range
One table per lab variable. Percentages are within each group (each column adds up to 100%), so the two groups can be compared despite their different sizes.

In [ ]:
def range_table(col):
    counts = pd.crosstab(raw[col], raw["group"]).reindex(ordered_levels(col), fill_value=0)
    pct = (100 * counts / counts.sum()).round(1)
    table = pd.concat({"n": counts, "%": pct}, axis=1)
    return table[[("n", "Non-CKD"), ("%", "Non-CKD"), ("n", "CKD"), ("%", "CKD")]]

tables = {col: range_table(col) for col in LAB_VARS}
for col, label in LAB_VARS.items():
    print(label)
    print(tables[col].to_string())
    print()

## Step 3: Grouped bar charts
Blue = Non-CKD, orange = CKD. Each chart is saved in the `figures` folder.

In [ ]:
for col, label in LAB_VARS.items():
    pct = tables[col]["%"]
    fig, ax = plt.subplots(figsize=(7, 3.6))
    grouped_bar(ax, pct, f"{label} by group", label)
    fig.savefig(f"figures/desc_{col}.png")
    plt.show()

## Step 4: One combined figure for the report
Four key kidney markers in one figure

In [ ]:
key = ["sc", "hemo", "sg", "al"]
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.flat, key):
    grouped_bar(ax, tables[col]["%"], LAB_VARS[col], "")
    ax.get_legend().remove()
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", ncol=2)
fig.suptitle("Key kidney markers: CKD vs non-CKD (% of patients in each group)",
             x=0.01, ha="left", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig("figures/desc_panel_key_markers.png")
plt.show()

## Step 4b: Headline figure for hospital leadership: share of patients with clearly abnormal results
Each marker is compared with a typical adult clinical limit. Because the data only has ranges, the cut-off is the nearest range boundary on the abnormal side, so these shares are conservative. The hospital lab's own reference ranges should be checked before using this in practice.

| Marker | Abnormal if | Clinical reference |
|---|---|---|
| Haemoglobin | below 11.3 g/dL | WHO anaemia threshold: below 12 g/dL (women) / 13 g/dL (men) |
| Urine albumin | dipstick 1+ or more | Albuminuria is a KDIGO marker of kidney damage |
| Urine specific gravity | 1.011 or lower | Low, fixed urine concentration (around 1.010) is typical in CKD |
| Blood urea | 48.1 mg/dL or higher | Above typical adult range (about 15–45 mg/dL) |
| Random glucose | 196 mg/dL or higher | ADA diabetes threshold: 200 mg/dL |
| Serum creatinine | 3.65 mg/dL or higher | About 3 times the typical upper limit (about 1.2 mg/dL) |

In [ ]:
ABNORMAL = {
    "Low haemoglobin (< 11.3 g/dL)": ("hemo", lambda code: code <= 4),
    "Albumin in urine (1+ or more)": ("al", lambda code: code >= 1),
    "Low urine specific gravity (≤ 1.011)": ("sg", lambda code: code <= 1),
    "High blood urea (≥ 48.1 mg/dL)": ("bu", lambda code: code >= 1),
    "High random glucose (≥ 196 mg/dL)": ("bgr", lambda code: code >= 3),
    "High serum creatinine (≥ 3.65 mg/dL)": ("sc", lambda code: code >= 1),
}
coded = d["all"]
groups = coded["class"].map({1: "CKD", 0: "Non-CKD"})
abnormal = pd.DataFrame({
    label: (100 * rule(coded[col]).groupby(groups).mean()).round(1)
    for label, (col, rule) in ABNORMAL.items()
}).T[["Non-CKD", "CKD"]].sort_values("CKD", ascending=False)
abnormal

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
grouped_barh(ax, abnormal, "Share of patients with clearly abnormal results")
fig.savefig("figures/desc_abnormal_profile.png")
plt.show()

## Step 5: Comorbidities and symptoms


In [ ]:
comorb = pd.DataFrame({
    g: [round(100 * raw.loc[raw["group"] == g, c].astype(int).mean(), 1) for c in CONDITIONS]
    for g in ["Non-CKD", "CKD"]
}, index=list(CONDITIONS.values()))
comorb

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
grouped_bar(ax, comorb, "Comorbidities and symptoms by group (none in non-CKD patients)", "")
ax.set_xticklabels(comorb.index, rotation=20, ha="right")
fig.savefig("figures/desc_comorbidities.png")
plt.show()

## Step 6: Summary table (Table 1 for the report)
Median range for each lab variable, and % with each condition, per group.



In [ ]:
def median_range(col, group):
    levels = ordered_levels(col)
    codes = raw.loc[raw["group"] == group, col].map({v: i for i, v in enumerate(levels)})
    return levels[int(codes.quantile(0.5, interpolation="lower"))]

rows = []
for col, label in LAB_VARS.items():
    rows.append({"Variable": label,
                 "Non-CKD (n=72)": median_range(col, "Non-CKD"),
                 "CKD (n=128)": median_range(col, "CKD")})
for col, label in CONDITIONS.items():
    rows.append({"Variable": f"{label}, %",
                 "Non-CKD (n=72)": comorb.loc[label, "Non-CKD"],
                 "CKD (n=128)": comorb.loc[label, "CKD"]})

table1 = pd.DataFrame(rows).set_index("Variable")
table1.to_csv("figures/table1_baseline_profile.csv")
table1

## Step 7: Observations
Two in three CKD patients have albumin in the urine, and more than half are clearly anaemic; both can be picked up with cheap tests (urine dipstick and haemoglobin).

1. CKD patients are older: median age range 59–66 years vs 43–51 for non-CKD, and 51% of CKD patients are 59 or older.
2. Anaemia is a strong feature of CKD: median haemoglobin 10–11.3 g/dL in CKD vs 13.9–15.2 g/dL in non-CKD; no non-CKD patient is below 11.3 g/dL.
3. Kidney damage shows in the urine: 66% of CKD patients have albumin grade 1 or higher (no non-CKD patient does), and CKD patients have more dilute urine (median specific gravity 1.015–1.017 vs 1.019–1.021).
4. Hypertension (61%) and diabetes (55%) are common in CKD patients; no non-CKD patient has either. Matching this, 40% of CKD patients have random glucose above 154 mg/dL, against none of the non-CKD group.
5. Blood pressure: the 0/1 diastolic flag differs only a little (raised in 47% of non-CKD vs 58% of CKD), but blood pressure category 2 occurs only in CKD patients (36%). The exact coding of these two variables should be confirmed in the UCI variable table.
6. Blood urea and serum creatinine are higher in CKD (urea above 48.1 mg/dL in 65% of CKD vs 12.5% of non-CKD), but the creatinine ranges are coarse: 68% of CKD patients still fall in the lowest range (< 3.65 mg/dL).

Data note: the non-CKD group has no comorbidities and no abnormal urine findings at all, so it looks like a healthy control group. This limits how far the comparison applies to a typical clinic population.

> *All lab values in this dataset are given as ranges, so we report counts, percentages and the median range per group. Means, standard deviations, histograms and boxplots (planned in the first presentation) cannot be calculated from ranges, so they are replaced by ordered bar charts of the ranges and by the share of patients outside clinical reference limits.*



## Step 8: Download all figures
A zip file with every chart and Table 1 downloads to your computer. Upload the files to the shared Google Drive folder.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("descriptive_figures", "zip", "figures")
files.download("descriptive_figures.zip")